In [1]:
import torch

We want to find the derivative of a simple parabola:


$$y = x^2$$

Manual Calculus: $\frac{dy}{dx} = 2x$. At $x=3$, the slope is $\mathbf{6}$.
PyTorch Autograd: Automates the chain rule using a computational graph.

In [4]:
def derivative_of_sq(X):
    return 2*X

In [5]:
dy_dx = derivative_of_sq(3)
print(dy_dx)

6


# *With Autograd*

In [11]:
X = torch.tensor(3.0, requires_grad = True)

Y = X**2
print(Y)

Y.backward()
print(X.grad)

tensor(9., grad_fn=<PowBackward0>)
tensor(6.)


We are exploring a composite function where:


$$y = x^2, \quad z = \sin(y) \implies z = \sin(x^2)$$

📐 The Calculus (Manual)

To find $\frac{dz}{dx}$, we apply the Chain Rule:


$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx} = \cos(y) \cdot 2x = \cos(x^2) \cdot 2x$$



In [13]:
import math
def derivative_of_comp_func(X):
    return math.cos(X**2) * 2 * X

In [14]:
dz_dx = derivative_of_comp_func(3)
print(dz_dx)

-5.466781571308061


# *With Autograd*

In [17]:
X = torch.tensor(3.0, requires_grad = True)
Y = X**2
Z = torch.sin(Y)

print(Z)
Z.backward()
print(X.grad)

tensor(0.4121, grad_fn=<SinBackward0>)
tensor(-5.4668)


***The Single Neuron: Manual vs. Autograd***

We are simulating a single neuron with one input ($x$), one weight ($w$), a bias ($b$), and a Binary Cross-Entropy (BCE) loss.

📐 1. The Manual Calculus

To update our parameters, we need the gradients $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$. Using the chain rule:

For the weight ($w$):


$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w} = (\hat{y} - y)x$$

For the bias ($b$):


$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial b} = (\hat{y} - y)(1) = \hat{y} - y$$

In [24]:
X = torch.tensor(3.0)
W = torch.tensor(2.0)
b = torch.tensor(1.0)
y = torch.tensor(0.0) # True Label

In [22]:
def binary_cross_entropy_loss(y,y_pred):
    epsilon = 1e-8
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    return -(y * torch.log(y_pred) + (1-y) * torch.log(1-y_pred))

In [25]:
Z = X*W + b
y_pred = torch.sigmoid(Z)
loss = binary_cross_entropy_loss(y,y_pred)
print(loss)

tensor(7.0010)


In [27]:
d_loss_dw = X*(y_pred-y) # this derivation can be found in the attachted pdf and sneakpeak is in markdown above.
d_loss_db = y_pred-y 

print(f"gradient w.r.t W: {d_loss_dw}")
print(f"gradient w.r.t b: {d_loss_db}")

gradient w.r.t W: 2.997267007827759
gradient w.r.t b: 0.9990890026092529


# *With Autograd*

In [28]:
X = torch.tensor(3.0)
W = torch.tensor(2.0, requires_grad = True)
b = torch.tensor(1.0, requires_grad = True)
y = torch.tensor(0.0) 

In [37]:
#forward pass
z = X*W + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y,y_pred)
print(loss)

tensor(7.0010, grad_fn=<NegBackward0>)


In [38]:
W.grad.zero_() # we use grad.zero_() inplace function so that the gradients dont accumulate u=in multiple runs. if we dont use it, previous value and current value will get added which we dont want.
b.grad.zero_()
loss.backward()
print(W.grad)
print(b.grad)

tensor(2.9973)
tensor(0.9991)
